# Text Processing and Labeling with Anthropic API

This notebook provides functions to:
1. Connect to the Anthropic API
2. Process and segment text files into context-aware chunks
3. Send chunks to a labeling agent with customizable instructions
4. Optionally maintain or ignore context between segments

In [3]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Union, Optional, Tuple
import anthropic
from anthropic import Anthropic
import tiktoken
from dotenv import load_dotenv

In [7]:
# Load API key from environment variables
# Initialize the Anthropic client
load_dotenv("./.env")
api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("ANTHROPIC_API_KEY environment variable is not set. Please set it in a .env file or directly in the environment.")

client = Anthropic(api_key=api_key)

## Text Segmentation Functions

In [8]:
# Initialize tokenizer for Claude models
tokenizer = tiktoken.get_encoding("cl100k_base")

In [9]:
def count_tokens(text: str) -> int:
    """Count the number of tokens in a string."""
    return len(tokenizer.encode(text))

In [10]:
def read_text_file(file_path: str) -> str:
    """Read text from a file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        raise Exception(f"Error reading file: {e}")

In [11]:
def segment_text_naive(text: str, max_tokens: int = 8000) -> List[str]:
    """Segment text naively by paragraphs up to max_tokens."""
    segments = []
    paragraphs = re.split(r'\n\s*\n', text)
    
    current_segment = ""
    current_token_count = 0
    
    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue
            
        paragraph_token_count = count_tokens(paragraph)
        
        # If a single paragraph exceeds max_tokens, split it by sentences
        if paragraph_token_count > max_tokens:
            if current_segment:
                segments.append(current_segment)
                current_segment = ""
                current_token_count = 0
                
            sentences = re.split(r'(?<=[.!?]) +', paragraph)
            temp_segment = ""
            temp_token_count = 0
            
            for sentence in sentences:
                sentence_token_count = count_tokens(sentence)
                
                if temp_token_count + sentence_token_count <= max_tokens:
                    temp_segment += sentence + " "
                    temp_token_count += sentence_token_count
                else:
                    if temp_segment:
                        segments.append(temp_segment.strip())
                    temp_segment = sentence + " "
                    temp_token_count = sentence_token_count
            
            if temp_segment:
                segments.append(temp_segment.strip())
        
        # Normal case: add paragraph to current segment if it fits
        elif current_token_count + paragraph_token_count <= max_tokens:
            current_segment += paragraph + "\n\n"
            current_token_count += paragraph_token_count + 2  # +2 for newlines
        else:
            segments.append(current_segment.strip())
            current_segment = paragraph + "\n\n"
            current_token_count = paragraph_token_count + 2
    
    if current_segment:
        segments.append(current_segment.strip())
    
    return segments

In [12]:
def segment_text_contextual(text: str, max_tokens: int = 8000, context_tokens: int = 1000) -> List[str]:
    """Segment text with context awareness by including overlap between segments."""
    segments = []
    paragraphs = re.split(r'\n\s*\n', text)
    
    # First create segments without context overlap
    current_segment = []
    current_token_count = 0
    paragraph_indices = []  # Track which paragraphs are in which segments
    
    for i, paragraph in enumerate(paragraphs):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
            
        paragraph_token_count = count_tokens(paragraph)
        
        # If a single paragraph exceeds the available tokens, split it by sentences
        if paragraph_token_count > max_tokens - context_tokens:
            if current_segment:
                segments.append("\n\n".join(current_segment))
                paragraph_indices.append(current_segment)
                current_segment = []
                current_token_count = 0
                
            sentences = re.split(r'(?<=[.!?]) +', paragraph)
            temp_segment = []
            temp_token_count = 0
            
            for sentence in sentences:
                sentence_token_count = count_tokens(sentence)
                
                if temp_token_count + sentence_token_count <= max_tokens - context_tokens:
                    temp_segment.append(sentence)
                    temp_token_count += sentence_token_count
                else:
                    if temp_segment:
                        segments.append(" ".join(temp_segment))
                        paragraph_indices.append([f"{i}:sentence"])
                    temp_segment = [sentence]
                    temp_token_count = sentence_token_count
            
            if temp_segment:
                segments.append(" ".join(temp_segment))
                paragraph_indices.append([f"{i}:sentence"])
        
        # Normal case: add paragraph to current segment if it fits
        elif current_token_count + paragraph_token_count <= max_tokens - context_tokens:
            current_segment.append(paragraph)
            current_token_count += paragraph_token_count + 2  # +2 for newlines
        else:
            if current_segment:
                segments.append("\n\n".join(current_segment))
                paragraph_indices.append(current_segment)
            current_segment = [paragraph]
            current_token_count = paragraph_token_count
    
    if current_segment:
        segments.append("\n\n".join(current_segment))
        paragraph_indices.append(current_segment)
    
    # Now add context to each segment
    contextual_segments = []
    for i, segment in enumerate(segments):
        prefix = ""
        suffix = ""
        
        # Add previous context if available
        if i > 0:
            prev_text = segments[i-1]
            # Take only the last part that fits within context_tokens
            prev_paragraphs = re.split(r'\n\s*\n', prev_text)
            prev_context = []
            prev_tokens = 0
            
            for p in reversed(prev_paragraphs):
                p_tokens = count_tokens(p)
                if prev_tokens + p_tokens <= context_tokens // 2:
                    prev_context.insert(0, p)
                    prev_tokens += p_tokens
                else:
                    break
            
            if prev_context:
                prefix = "Previous Context:\n" + "\n\n".join(prev_context) + "\n\nCurrent Segment:\n"
        
        # Add following context if available
        if i < len(segments) - 1:
            next_text = segments[i+1]
            # Take only the first part that fits within context_tokens
            next_paragraphs = re.split(r'\n\s*\n', next_text)
            next_context = []
            next_tokens = 0
            
            for p in next_paragraphs:
                p_tokens = count_tokens(p)
                if next_tokens + p_tokens <= context_tokens // 2:
                    next_context.append(p)
                    next_tokens += p_tokens
                else:
                    break
            
            if next_context:
                suffix = "\n\nFollowing Context:\n" + "\n\n".join(next_context)
        
        contextual_segments.append(prefix + segment + suffix)
    
    return contextual_segments

In [13]:
def segment_text_semantic(text: str, max_tokens: int = 8000, model: str = "claude-3-haiku-20240307") -> List[str]:
    """Use Claude to intelligently segment text based on semantic boundaries."""
    # Naive initial segmentation to ensure we're under token limits
    initial_segments = segment_text_naive(text, max_tokens=16000)
    final_segments = []
    
    for segment in initial_segments:
        if count_tokens(segment) <= max_tokens:
            final_segments.append(segment)
            continue
        """"
        Would it be better to pass the text to the LLM, then to the tokenizer and split based on that?
        Compare with 'ground truth'. How did Kajal tokenize?
        Iterative labeling? Pass text -> read it once -> Build 10-20 segments -> assign labels -> break down into tokens -> label with semantic context
        Alternative - Create segments once then use the read all then label approach - x50 - 100?
        Calculate Label Entropy
        """

        # If segment is too large, ask Claude to divide it
        prompt = f"""
        I need you to divide the following text into {max_tokens//2000}-{max_tokens//1000} coherent segments. 
        Make sure each segment is semantically coherent and ends at a natural break point such as the end of a section, 
        a scene change, or a logical conclusion to a thought. Do not cut in the middle of sentences or paragraphs unless necessary.
        Only return the segmented text with no additional commentary. Place the string <SEGMENT_BREAK> between each segment.
        
        Text to segment:
        {segment}
        """
        
        try:
            response = client.messages.create(
                model=model,
                max_tokens=1024,
                temperature=0,
                system="You are a helpful professional Linguist that segments text at natural boundaries.",
                messages=[{"role": "user", "content": prompt}]
            )
            
            sub_segments = response.content[0].text.split("<SEGMENT_BREAK>")
            final_segments.extend([s.strip() for s in sub_segments if s.strip()])
        except Exception as e:
            print(f"Error during semantic segmentation: {e}")
            # Fallback to naive segmentation - not used as of now
            # sub_segments = segment_text_naive(segment, max_tokens)
            # final_segments.extend(sub_segments)
            break
            
    return final_segments

## Text Labeling Functions

In [14]:
def label_segment(segment: str, 
                 labeling_instructions: str, 
                 label_categories: Optional[List[str]] = None,
                 model: str = "claude-3-opus-20240229",
                 temperature: float = 0.3) -> Dict:
    """Label a single text segment using Claude."""
    
    system_prompt = "You are an expert text annotator that provides accurate labels and analyses. You strictly adhere to format instructions given by the user. If they do not want explanatory text you will not provide any."
    
    if label_categories:
        categories_str = "\n".join([f"- {cat}" for cat in label_categories])
        user_prompt = f"""
        Please analyze and label the following text segment according to these instructions:
        
        {labeling_instructions}
        
        Use only the following label categories:
        {categories_str}
        
        Format your response as a JSON object with these fields:
        - "labels": A list of labels that apply to this segment
        - "confidence": A number from 0-1 indicating your confidence in each label
        - "explanation": A brief explanation of why these labels apply
        
        Text segment to label:
        """
    else:
        user_prompt = f"""
        Please analyze and label the following text segment according to these instructions:
        
        {labeling_instructions}
        
        Format your response as a JSON object with these fields:
        - "labels": A list of labels that apply to this segment
        - "confidence": A number from 0-1 indicating your confidence in each label
        - "explanation": A brief explanation of why these labels apply
        
        Text segment to label:
        """
    
    user_prompt += "\n" + segment
    
    try:
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}]
        )
        
        # Extract JSON from the response
        response_text = response.content[0].text
        
        # Try to extract JSON
        try:
            # Try to find JSON pattern in the response
            json_match = re.search(r'\{[\s\S]*\}', response_text)
            if json_match:
                json_str = json_match.group(0)
                result = json.loads(json_str)
            else:
                # Fallback: Ask Claude to reformat as JSON
                fix_prompt = f"Convert your previous response to valid JSON:"
                fix_response = client.messages.create(
                    model="claude-3-haiku-20240307",
                    max_tokens=1024,
                    temperature=0,
                    system="You format text as valid JSON.",
                    messages=[
                        {"role": "user", "content": user_prompt},
                        {"role": "assistant", "content": response_text},
                        {"role": "user", "content": fix_prompt}
                    ]
                )
                json_match = re.search(r'\{[\s\S]*\}', fix_response.content[0].text)
                json_str = json_match.group(0) if json_match else "{}"
                result = json.loads(json_str)
        except Exception as e:
            print(f"Error parsing JSON response: {e}")
            result = {
                "labels": ["ERROR_PARSING_RESPONSE"],
                "confidence": 0,
                "explanation": f"Failed to parse response: {str(e)}",
                "raw_response": response_text
            }
            
        return result
    
    except Exception as e:
        print(f"API call error: {e}")
        return {
            "labels": ["API_ERROR"],
            "confidence": 0,
            "explanation": f"API error: {str(e)}"
        }

In [15]:
def label_segments(segments: List[str], 
                  labeling_instructions: str, 
                  label_categories: Optional[List[str]] = None,
                  context_aware: bool = False,
                  model: str = "claude-3-opus-20240229",
                  temperature: float = 0.5) -> List[Dict]:
    """Label multiple text segments with option to maintain context awareness."""
    results = []
    
    if not context_aware:
        # Process segments independently
        for i, segment in enumerate(segments):
            print(f"Processing segment {i+1}/{len(segments)}...")
            result = label_segment(
                segment, 
                labeling_instructions, 
                label_categories,
                model,
                temperature
            )
            results.append(result)
            # Add small delay to avoid rate limits
            time.sleep(0.5)
    else:
        # Context-aware processing
        system_prompt = "You are an expert text annotator that provides accurate labels and analyses."
        conversation = []
        
        # Initial setup message
        if label_categories:
            categories_str = "\n".join([f"- {cat}" for cat in label_categories])
            setup_msg = f"""
            You will analyze a series of text segments according to these instructions:
            
            {labeling_instructions}
            
            Use only the following label categories:
            {categories_str}
            
            For each segment, format your response as a JSON object with these fields:
            - "labels": A list of labels that apply to this segment
            - "confidence": A number from 0-1 indicating your confidence in each label

            You can consider previously analyzed segment for context.
            """
           # - "explanation": A brief explanation of why these labels apply
            
           # You can consider previously analyzed segments for context.
           # """
        else:
            setup_msg = f"""
            You will analyze a series of text segments according to these instructions:
            
            {labeling_instructions}
            
            For each segment, format your response as a JSON object with these fields:
            - "labels": A list of labels that apply to this segment
            - "confidence": A number from 0-1 indicating your confidence in each label

                        You can consider previously analyzed segment for context.
            """
            #- "explanation": A brief explanation of why these labels apply
            
            #You can consider previously analyzed segments for context.
            #"""
        
        conversation.append({"role": "user", "content": setup_msg})
        
        # Get acknowledgment from the model
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            temperature=temperature,
            system=system_prompt,
            messages=conversation
        )
        
        ack_response = response.content[0].text
        conversation.append({"role": "assistant", "content": ack_response})
        
        # Process each segment in order
        for i, segment in enumerate(segments):
            print(f"Processing segment {i+1}/{len(segments)} with context...")
            segment_msg = f"Segment {i+1}:\n\n{segment}"
            conversation.append({"role": "user", "content": segment_msg})
            
            response = client.messages.create(
                model=model,
                max_tokens=1024,
                temperature=temperature,
                system=system_prompt,
                messages=conversation
            )
            
            response_text = response.content[0].text
            conversation.append({"role": "assistant", "content": response_text})
            
            # Extract JSON from the response
            try:
                json_match = re.search(r'\{[\s\S]*\}', response_text)
                if json_match:
                    json_str = json_match.group(0)
                    result = json.loads(json_str)
                else:
                    # Ask for JSON format
                    fix_prompt = "Please reformat your last response as a valid JSON object."
                    conversation.append({"role": "user", "content": fix_prompt})
                    
                    fix_response = client.messages.create(
                        model=model,
                        max_tokens=1024,
                        temperature=0,
                        system=system_prompt,
                        messages=conversation
                    )
                    
                    fix_text = fix_response.content[0].text
                    conversation.append({"role": "assistant", "content": fix_text})
                    
                    json_match = re.search(r'\{[\s\S]*\}', fix_text)
                    json_str = json_match.group(0) if json_match else "{}"
                    result = json.loads(json_str)
            except Exception as e:
                print(f"Error parsing JSON response: {e}")
                result = {
                    "labels": ["ERROR_PARSING_RESPONSE"],
                    "confidence": 0,
                    "explanation": f"Failed to parse response: {str(e)}",
                    "raw_response": response_text
                }
            
            results.append(result)
            time.sleep(1)  # Slightly longer delay for context-aware processing
    
    return results

In [16]:
def export_results(segments: List[str], 
                  labels_results: List[Dict], 
                  output_file: str = "labeled_segments.json") -> None:
    """Export segmentation and labeling results to a file."""
    output = []
    
    for i, (segment, result) in enumerate(zip(segments, labels_results)):
        output.append({
            "segment_id": i+1,
            "text": segment,
            "labels": result.get("labels", []),
            "confidence": result.get("confidence", 0),
            "explanation": result.get("explanation", "")
        })
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)
        
    print(f"Results exported to {output_file}")
    
    # Also export as CSV for easy viewing
    csv_file = output_file.replace(".json", ".csv")
    df = pd.DataFrame(output)
    df.to_csv(csv_file, index=False)
    print(f"Results also exported to {csv_file}")

## Complete Text Processing and Labeling Pipeline

In [22]:
def process_text_file(file_path: str,
                     labeling_instructions: str,
                     label_categories: Optional[List[str]] = None,
                     max_tokens_per_segment: int = 8000,
                     segmentation_method: str = "contextual",  # "naive", "contextual", or "semantic"
                     context_aware_labeling: bool = False,
                     labeling_model: str = "",
                     temperature: float = 0.3,
                     output_file: Optional[str] = None) -> Tuple[List[str], List[Dict]]:
    """Complete pipeline to process and label a text file."""
    
    # Read the text file
    print(f"Reading file: {file_path}")
    text = read_text_file(file_path)

    print("Skipping segmentation...")
    # Segment the text
    #print(f"Segmenting text using {segmentation_method} method...")
    #if segmentation_method == "naive":
    #    segments = segment_text_naive(text, max_tokens=max_tokens_per_segment)
    #elif segmentation_method == "contextual":
    #    segments = segment_text_contextual(text, max_tokens=max_tokens_per_segment)
    #elif segmentation_method == "semantic":
    #    segments = segment_text_semantic(text, max_tokens=max_tokens_per_segment)
    #else:
    #    raise ValueError(f"Unknown segmentation method: {segmentation_method}")
    
    #print(f"Text segmented into {len(segments)} segments")
    
    # Label the segments
    print(f"Labeling segments {'with context awareness' if context_aware_labeling else 'independently'}...")
    results = label_segments(
        segments, 
        labeling_instructions,
        label_categories,
        context_aware=context_aware_labeling,
        model=labeling_model,
        temperature=temperature
    )
    
    # Export results if output file is specified
    if output_file:
        export_results(segments, results, output_file)
    else:
        # Generate default output filename based on input file
        input_name = Path(file_path).stem
        default_output = f"{input_name}_labeled.json"
        export_results(segments, results, default_output)
    
    return segments, results

## Example Usage

In [18]:
# Example usage with sample labeling instructions and categories
sample_file_path = "./story_fulltext.txt"  # Update with actual file path

labeling_instructions = """
Analyze each text segment for emotional content and narrative elements.
Identify the predominant emotions expressed or evoked by the text. The text is given as a series of tokens formatted like this: ["text"].
Analyze each text segment for the acting characters, the location, the time, and emotional content. If a label is not applicable to a token do not make up a label but say 0 or null. Also provide a score between 0 and 1 how confident you are in the label.
Format your response like this:
token: {input token to be labeled}
location: {your location label} {confidence score}
time: {your time label} {confidence score}
emotions: {your emotion label} {confidence score}
characters: {your character label} {confidence score}

Example:
token: ["Rachel smiled at Jack."]
location: [null] - 1
time: [null] - 1
emotion: [friendly] - 0.6
character: [Rachel, Jack] - 0.9

Explanation:
The example has not context outside the token. So there is no location or time specified. Therefore no label can be assigned to these categories with high certainty. The smile is interpreted as friendly with a confidence of 0.6. Since there is no context this is ambiguous as a smile can mean a lot of things. Even with context it might still not be a 100% certainty. The Characters are Rachel and Jack with high probability as we explicitly hear from them. However Jack might not be in the scene, therefore not 100% certainty but 95%.
"""

label_categories = [
    "Characters", "Location", "Time", "emotion"
]

# Uncomment to run the full pipeline
# segments, results = process_text_file(
#     file_path=sample_file_path,
#     labeling_instructions=labeling_instructions,
#     label_categories=label_categories,
#     segmentation_method="contextual",
#     context_aware_labeling=True,
#     labeling_model="claude-3-sonnet-20240229"
# )

In [20]:
file_path = Path("./tokens.csv")
labeling_instructions = labeling_instructions
label_categories = label_categories
segmentation_method = "semantic"
context_aware_labeling = True
labeling_model = "claude-3-7-sonnet-20250219"
segments, results = process_text_file(file_path=file_path, labeling_instructions=labeling_instructions, label_categories=label_categories, segmentation_method="semantic", context_aware_labeling=True, labeling_model=labeling_model)

Reading file: story_fulltext.txt
Segmenting text using semantic method...
Text segmented into 1 segments
Labeling segments with context awareness...
Processing segment 1/1 with context...
Results exported to story_fulltext_labeled.json
Results also exported to story_fulltext_labeled.csv


## Custom Label Categories

In [ ]:
# Example with custom neuroscience-related labels
neuro_labeling_instructions = """
Analyze each text segment for the acting characters, the location, the time, emotions conveyed. If a label is not applicable to a token do not make up a label but say 0 or null. Also provide a score between 0 and 1 how confident you are in the label.
Format your response like this:
token: {input token to be labeled}
location: {your location label} {confidence score}
time: {your time label} {confidence score}
emotions: {your emotion label} {confidence score}
characters: {your character label} {confidence score}

Example:
token: [Rachel smiled at Jack.]
location: [null] - 1
time: [null] - 1
emotion: [friendly] - 0.6
character: [Rachel, Jack] - 0.9

Explanation:
The example has not context outside the token. So there is no location or time specified. Therefore no label can be assigned to these categories with high certainty. The smile is interpreted as friendly with a confidence of 0.6. Since there is no context this is ambiguous as a smile can mean a lot of things. Even with context it might still not be a 100% certainty. The Characters are Rachel and Jack with high probability as we explicitly hear from them. However Jack might not be in the scene, therefore not 100% certainty but 95%.
"""

#neuro_label_categories = [
    # Story Categories
    
    # Cognitive processes - maybe to explore a potential mapping between stimulus and response -- alignment in latent?
    #"Memory", "Attention", "Emotion", "Decision Making", "Language", "Perception",
    #"Learning", "Executive Function", "Social Cognition", "Motor Control",

#]

# Uncomment to run with neuroscience-focused labeling
# segments, results = process_text_file(
#     file_path=sample_file_path,
#     labeling_instructions=neuro_labeling_instructions,
#     label_categories=neuro_label_categories,
#     segmentation_method="semantic",
#     context_aware_labeling=False,
#     labeling_model="claude-3-opus-20240229"
# )